# **Mini-Project A2 Phase 1: Image Mask Manipulation with Semantic Segmentation**

In this project, you will learn to use segmentation and perform three major manipulatons to images, (1) Inpainting, (2) Styling, and (3) Replacing.

Please follow the notebook in order. Make sure to add code where indicated by `''' TODO '''` or `# YOUR CODE HERE`. Please answer all questions indicated with '❓', and answer where it says 'ANSWER HERE'.

Ensure your notebook is easy to follow. All written report answers should be provided in the notebook itself.

## **Overview**



1.   Segmentation
2.   Inpainting
3.   Styling
4.   Replacement


This lab will use 2 images, please add the paths accordingly! The TA will change and use their own images when grading.

Please use image1 and image2 throughout code so TAs can easily change images at top here!



In [ ]:
from PIL import Image
# Image1 and Image 2 paths, please use your own images!!

image1_path = '''TODO'''
image2_path = '''TODO'''

image1 = Image.open(image1_path).convert("RGB")
image2 = Image.open(image2_path).convert("RGB")

# **1. Segmentation**

We provide you with the code to run semantic segmentation using [DeepLabV3](https://pytorch.org/hub/pytorch_vision_deeplabv3_resnet101/) in pytorch below.

Note: You can run this without GPU, it will not be considerably slower since we are only doing inference on max 2 images this lab!!

We also provide the class names list below for masks. There are total 20 objects.

In [ ]:
from torchvision import transforms
from PIL import Image
import torch
from torchvision.models.segmentation import deeplabv3_resnet50

def segmentation(image):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    image_tensor = transform(image).unsqueeze(0).cuda()
    model = deeplabv3_resnet50(pretrained=True).cuda().eval()

    with torch.no_grad():
        output = model(image_tensor)['out'][0]
        segmentation_mask = output.argmax(0).detach().cpu().numpy()

    return segmentation_mask

class_names = [
    "Background",
    "Aeroplane",
    "Bicycle",
    "Bird",
    "Boat",
    "Bottle",
    "Bus",
    "Car",
    "Cat",
    "Chair",
    "Cow",
    "Dining Table",
    "Dog",
    "Horse",
    "Motorbike",
    "Person",
    "Potted Plant",
    "Sheep",
    "Sofa",
    "Train",
    "TV/Monitor"
]

## **1.1 Visualization**

Like previous labs, you must visualize the outputs and images!!

Please complete `display_masks()` to show the original image, and all the masks detected, with their class names on top.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2


def display_masks(image, class_names):
    '''
    TODO
    '''

In [ ]:
# Show image and all detected masks with class names (exclude background)
display_masks(image1, segmentation_mask, class_indices)

In [ ]:
display_masks(image1, segmentation_mask, class_indices)

# **2. Inpainting: Object Removal**




Inpainting refers to the process of removing objects from an image by filling the corresponding regions (defined by a segmentation mask) with information, typically the surrounding background. This helps in maintaining visual continuity and gives the essence of object removal.

In this section, you will complete four functions, each implementing a different type of inpainting. Each method will replace the region defined by the segmentation mask, (given a class) with a unique technique.

Common Instructions:
- All functions should take in the image (PIL Image), and class name (as string) to inpaint.
- You can add additional parameters but set defaults!
- You can use OpenCV functions for various parts, the isntructions mention which ones are allowed. For any others, please confirm with TAs!

---

## **2.1 Mean Color Fill**

Complete `mean_color_fill()`.

**Description:** Replace the region defined by the segmentation mask with the mean RGB value of the surrounding region.

**Steps:**
1. Identify the pixels around the segmentation mask (the neighboring background).
2. Compute either: (a) Mean Color: Calculate the average RGB value of the surrounding pixels. (b) Majority Vote Color: Use the most frequent color in the surrounding pixels.
3. Replace all pixels in the mask with this computed value.



In [ ]:
def mean_color_fill(image, class_name):
    '''
    TODO
    '''

## **2.2 Noise Fill**

Complete `noise_fill()`.

**Description:** Replace the masked region with a blurred and feathered version of the surrounding area to achieve smooth blending.

**Steps:**
1. Convert the segmentation mask into a binary mask for the selected class_index.
2. Feather the mask by applying dilation, expanding its edges to include some surrounding regions.
3. Create a blurred version of the image using a Gaussian blur with a large kernel size.
4. Blend the original image with the blurred version in the masked region, using the feathered mask as a weighting factor.

You may use `cv2.dilate()` for feathering and `cv2.GaussianBlur()` for blurring.

In [ ]:
def noise_fill(image, class_name):
    '''
    TODO
    '''

## **2.3 Linear Interpolation**

Complete `linear_interpolation()`. For this, you can use `cv2.inpaint()` opencv in-built function (see [documentation](https://docs.opencv.org/3.4/df/d3d/tutorial_py_inpainting.html)) or implement your own! Steps to implement are below.

**Description:** Blend the colors of the region edges to fill the gap with a linear interpolation.

**Steps:**
1. Identify the boundary of the segmentation mask and the pixel values along the edges.
2. Interpolate between these edge pixel values to estimate the colors for the missing region.
3. Fill the masked region with these interpolated values.

In [ ]:
def linear_interpolation(image, class_name):
    '''
    TODO
    '''

## **2.4 Patch Inpaint**

Complete `patch_inpaint()`.

**Description:** Fill the masked region by replicating small patches from the surrounding area.

**Steps:**
1. Extract a small patch (e.g., 3x3 or 5x5) from the area around the mask. Patch size must be input to `patch_inpaint()`.
2. Start filling the mask from either:
(a) The top of the region, moving right, or
(b) The bottom, moving left.
3. After filling one row (or column) of the mask, extract a new patch from the surrounding area and repeat.

In [ ]:
def patch_inpaint(image, class_name):
    '''
    TODO
    '''

## **2.5 Visualize Inpainted images**

Please make a function, `visualize_inpaint()` to visualize all inpainted images side-by-side. The function should be able to take in an image, and class index to inpaint select class.

In [ ]:
def visualize_inpaint(image, class_name):
    '''
    TODO
    '''

In [ ]:
visualize_inpaint(image1, "Person")

In [ ]:
visualize_inpaint(image2, "Person")

### ❓ **Q 2.5 Explain the differences between all inpaint methods, and their advantages and disadvantages.**

ANSWER HERE

# **3. Image Styling**

In this section, we will try to add styles to all unique segmentation masks!

Write a function called `apply_styles()` which takes an image as input and applies unique styles to each distinct **instance** in the mask. There can be multiple instances in a class!

You can specify styles like pixelation, invert, and more for the classes. You can use OpenCV to find styles you like. Make them look cool!

**Please ensure you have at least 4 unique styles and demonstrate them on your image by visualizing the output!**

*Hint*: you can use `cv2.connectedComponents()`.

In [ ]:
import numpy as np
import cv2
from PIL import Image
import random

def apply_styles(image, styles=None):
    '''
    TODO
    '''

In [ ]:
# Run on image 1
apply_styles(image1)

In [ ]:
# run on image 2
apply_styles(image2)

# **4. Replacement**

## **4.1 Replacement with semantic masks**

Write a function `replace_object()` which takes in two images, `image1` and `image2` and uses "copy paste" replacement.

**Goal:** Your goal is, given both images have at least one mask of same class, we will remove the class from `image1` (use one of the inpaint functions from Section 2) and replace it with a randomly chosen object of same class from `image2`.

- You must ensure both images have the class you want to replace.
- Your image2 masked must be resized appropriately to fit the inpainted region of image1!

- You may use OpenCV functions where necesary. Please confirm with TA the OpenCV functions you are using.

**Output:** Your output must be the modified image!

**Visualization:** Please show image1, image2 and replaced_image.

In [ ]:
def replace_object(image1, image2, segmentation, class_name):
    '''
    TODO
    '''
    return replaced_image

In [ ]:
replaced_image = replace_object(image1, image2, "Person")
# show all 3: image1, image2, and replaced_image

## **4.2 More visualizations!**

Please make another function (`show_intermediate()`) to show all intermediate images from `replace_object()`. This includes, original `image1` and `image2`, inpainted `image1`, the selected masked region you chose from `image2`, and finally the output image.

In [ ]:
def show_intermediate():
    '''
    TODO
    '''

## ❓ **Q 4.3 Explain your replacement code.**

ANSWER HERE

# End of Phase 1 😊🥳